In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import datetime
import requests
import json
import numpy as np
import ast


## Read in Data (csv has omdb data)

In [2]:
movies = pd.read_csv("eng_movies_gt_2000_with_omdb_dataset_102023.csv", low_memory=False)
movies_cleaned = pd.read_csv("eng_movies_gt_2000_with_omdb_dataset_102023.csv", low_memory=False)

In [3]:
missing_movies = movies.isna()  # or df.isnull()
missing_movies.sum()

id                            0
adult                         0
backdrop_path            177362
belongs_to_collection    247368
budget                        0
genres                        0
homepage                 193044
id.1                          0
imdb_id                   84558
original_language             0
original_title                4
overview                      0
popularity                    0
poster_path               62905
production_companies          0
production_countries          0
release_date                  0
revenue                       0
runtime                       0
spoken_languages              0
status                        0
tagline                  195252
title                         4
video                         0
vote_average                  0
vote_count                    0
keywords                      0
omdb_Title                74724
omdb_Year                 74724
omdb_Rated               185781
omdb_Released             97483
omdb_Run

### Clean Genre

* Find out how many rows with genre are empty (i.e. [], Null, None, NaN, etc.)
* Replace [] with NaN values so we can universally replace them if possible
* Change the format of tmdb "genres" column to match omdb "genres" column
* Replace the empty tmdb genres with omdb_Genre
* This saves us around 44,427 Movies which previously had empty genres

### Clean Keywords
* Replace [] with NaN values 
* change format from list of dictionaries to string separated by comma


In [4]:
# Find rows where genre is null/NaN/[] etc.
print(" Number of rows with []: " + str(len(movies[movies["genres"] == "[]"])))
print(" Number of rows with NaN: " + str(len(movies[movies["genres"] == np.nan])))
print(" Number of rows with None: " + str(len(movies[movies["genres"] == None])))
print(" Number of rows with '': " + str(len(movies[movies["genres"] == ''])))

 Number of rows with []: 79723
 Number of rows with NaN: 0
 Number of rows with None: 0
 Number of rows with '': 0


In [5]:
movies_cleaned["genres"] = movies_cleaned["genres"].apply(ast.literal_eval)
movies_cleaned["keywords"] = movies_cleaned["keywords"].apply(ast.literal_eval)

In [6]:
movies_cleaned["genres"][3]

[{'id': 16, 'name': 'Animation'}, {'id': 10751, 'name': 'Family'}]

In [7]:
#movies_cleaned['genres'] = movies_cleaned['genres'].replace([], np.NaN)
movies_cleaned['genres'] = movies_cleaned['genres'].apply(lambda y: np.nan if len(y)==0 else y)

In [8]:
movies_cleaned["genres"]

0                                                       NaN
1                         [{'id': 10751, 'name': 'Family'}]
2                       [{'id': 99, 'name': 'Documentary'}]
3         [{'id': 16, 'name': 'Animation'}, {'id': 10751...
4         [{'id': 18, 'name': 'Drama'}, {'id': 80, 'name...
                                ...                        
252745                                                  NaN
252746                                                  NaN
252747                                                  NaN
252748                  [{'id': 99, 'name': 'Documentary'}]
252749                  [{'id': 99, 'name': 'Documentary'}]
Name: genres, Length: 252750, dtype: object

In [9]:
#movies_cleaned['keywords'] = movies_cleaned['keywords'].replace('[]', np.NaN)
movies_cleaned['keywords'] = movies_cleaned['keywords'].apply(lambda y: np.nan if len(y)==0 else y)

In [10]:
type(movies_cleaned["keywords"][0])

list

In [11]:
def convert_list_of_dict_to_str(l):
    s = ''
    if  type(l) is list:
        for d in l:
            if "name" in d:
                s += d["name"] + ", "
        return s[:-2]
    else:
        return np.nan

In [12]:
movies_cleaned['genres'] = movies_cleaned['genres'].map(lambda x: convert_list_of_dict_to_str(x))

In [13]:
movies_cleaned["genres"].isna().sum()

79723

In [14]:
movies_cleaned.genres = movies_cleaned['genres'].fillna(movies_cleaned['omdb_Genre'])

In [15]:
movies_cleaned["genres"].isna().sum()

35296

In [16]:
movies_cleaned['keywords'] = movies_cleaned['keywords'].map(lambda x: convert_list_of_dict_to_str(x))

In [17]:
movies_cleaned["keywords"].isna().sum()

175080

In [18]:
movies_cleaned["keywords"]

0                                   sports, mountain biking
1                                                       NaN
2                                                megacities
3         parent child relationship, sydney, australia, ...
4         individual, dancing, robbery, factory worker, ...
                                ...                        
252745                                                  NaN
252746                                                  NaN
252747                                                  NaN
252748                                                  NaN
252749                                                  NaN
Name: keywords, Length: 252750, dtype: object

In [19]:
# OMDB Saved about 79,723 - 35,296 = 44,427

In [20]:
missing_movies_cleaned = movies_cleaned.isna()  # or df.isnull()
missing_movies_cleaned.sum()

id                            0
adult                         0
backdrop_path            177362
belongs_to_collection    247368
budget                        0
genres                    35296
homepage                 193044
id.1                          0
imdb_id                   84558
original_language             0
original_title                4
overview                      0
popularity                    0
poster_path               62905
production_companies          0
production_countries          0
release_date                  0
revenue                       0
runtime                       0
spoken_languages              0
status                        0
tagline                  195252
title                         4
video                         0
vote_average                  0
vote_count                    0
keywords                 175080
omdb_Title                74724
omdb_Year                 74724
omdb_Rated               185781
omdb_Released             97483
omdb_Run

In [21]:
movies_df = movies_cleaned[movies_cleaned['genres'].notna()]
# columns of interest
# omdb_plot, genre, keywords, overview

# Idea: concat plot, keywwrds, overview

In [22]:
print(movies_df.isna().sum())

id                            0
adult                         0
backdrop_path            146277
belongs_to_collection    212407
budget                        0
genres                        0
homepage                 162648
id.1                          0
imdb_id                   56866
original_language             0
original_title                4
overview                      0
popularity                    0
poster_path               48555
production_companies          0
production_countries          0
release_date                  0
revenue                       0
runtime                       0
spoken_languages              0
status                        0
tagline                  162948
title                         4
video                         0
vote_average                  0
vote_count                    0
keywords                 142021
omdb_Title                40596
omdb_Year                 40596
omdb_Rated               150663
omdb_Released             62983
omdb_Run

In [23]:
# Find rows where genre is null/NaN/[] etc. for overview
print(" Number of rows with None: " + str(len(movies_df[movies_df.overview == None])))
print(" Number of rows with '': " + str(len(movies_df[movies_df.overview == ''])))
print(" Number of rows with '': " + str(len(movies_df[movies_df.overview == '[]'])))

 Number of rows with None: 0
 Number of rows with '': 0
 Number of rows with '': 0


In [24]:
movies_df.overview

1         "Elmo is making a very, very super special sur...
2         "Timo Novotny labels his new project an experi...
3         "Nemo, an adventurous young clownfish, is unex...
4         "Selma, a Czech immigrant on the verge of blin...
5         "In an attempt to pull her family together, Ad...
                                ...                        
252742    "\u201cA Significant Name\u201d tells the stor...
252744    "\u2018Cinegraffic Score\u2019 is a 6 meter co...
252747    "A kaleidoscope of colors and foliage expresse...
252748    "Dr. Heiser explores the connection between UF...
252749    "They do accounting, handle human-resources ma...
Name: overview, Length: 217454, dtype: object

In [25]:
#keywords
print(" Number of rows with None: " + str(len(movies_df[movies_df.keywords == None])))
print(" Number of rows with '': " + str(len(movies_df[movies_df.keywords == ''])))
print(" Number of rows with '': " + str(len(movies_df[movies_df.keywords == '[]'])))

 Number of rows with None: 0
 Number of rows with '': 0
 Number of rows with '': 0


In [26]:
movies_df['keywords']

1                                                       NaN
2                                                megacities
3         parent child relationship, sydney, australia, ...
4         individual, dancing, robbery, factory worker, ...
5         suicide, paradise, child abuse, sea, loss of l...
                                ...                        
252742                                                  NaN
252744                                                  NaN
252747                                                  NaN
252748                                                  NaN
252749                                                  NaN
Name: keywords, Length: 217454, dtype: object

In [27]:
#plot
print(" Number of rows with None: " + str(len(movies_df[movies_df.omdb_Plot == None])))
print(" Number of rows with '': " + str(len(movies_df[movies_df.omdb_Plot == ''])))
print(" Number of rows with '': " + str(len(movies_df[movies_df.omdb_Plot == '[]'])))

 Number of rows with None: 0
 Number of rows with '': 0
 Number of rows with '': 0


In [28]:
movies_df.omdb_Plot

1         Elmo is making a super special surprise card f...
2         A remix of images and sounds, using a films or...
3         After his son is captured in the Great Barrier...
4         An Eastern European US immigrant with a love f...
5         In mourning over the tragic drowning of their ...
                                ...                        
252742                                                  NaN
252744                                                  NaN
252747    A year after a devastating flood has killed fi...
252748    Decades after surviving the Nostromo incident,...
252749                                                  NaN
Name: omdb_Plot, Length: 217454, dtype: object

In [29]:
#tagline
print(" Number of rows with None: " + str(len(movies_df[movies_df.tagline == None])))
print(" Number of rows with '': " + str(len(movies_df[movies_df.tagline == ''])))
print(" Number of rows with '': " + str(len(movies_df[movies_df.tagline == '[]'])))

 Number of rows with None: 0
 Number of rows with '': 0
 Number of rows with '': 0


In [30]:
movies_df.tagline

1                                                       NaN
2                                       A Megacities remix.
3         There are 3.7 trillion fish in the ocean. They...
4                               You don't need eyes to see.
5                    One of the living for one of the dead.
                                ...                        
252742                                                  NaN
252744                                                  NaN
252747                                                  NaN
252748    What is the connection between UFOs, alien abd...
252749                                                  NaN
Name: tagline, Length: 217454, dtype: object

In [31]:
# Doesn't work
cols = ['omdb_Plot', 'overview', 'keywords', 'tagline']
movies_df['plot_overview_keywords_tagline'] = movies_df[cols].apply(lambda row: '_'.join(row.values.astype(str)), axis=1)

/tmp/ipykernel_3575/3923073566.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  movies_df['plot_overview_keywords_tagline'] = movies_df[cols].apply(lambda row: '_'.join(row.values.astype(str)), axis=1)


In [36]:
movies_df['plot_overview_keywords_tagline'] = movies_df['plot_overview_keywords_tagline'].str.replace('nan_', '')

/tmp/ipykernel_3575/624805859.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  movies_df['plot_overview_keywords_tagline'] = movies_df['plot_overview_keywords_tagline'].str.replace('nan_', '')


In [37]:
movies_df['plot_overview_keywords_tagline'] = movies_df['plot_overview_keywords_tagline'].str.replace('_nan', '')

/tmp/ipykernel_3575/3063992768.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  movies_df['plot_overview_keywords_tagline'] = movies_df['plot_overview_keywords_tagline'].str.replace('_nan', '')


In [44]:
movies_df['plot_overview_keywords_tagline'][100]

'Two ambitious girls, despite their parents\' wishes, have their hearts set on careers in professional football._"Jess Bhamra, the daughter of a strict Indian couple in London, is not permitted to play organized soccer, even though she is 18. When Jess is playing for fun one day, her impressive skills are seen by Jules Paxton, who then convinces Jess to play for her semi-pro team. Jess uses elaborate excuses to hide her matches from her family while also dealing with her romantic feelings for her coach, Joe."_london, england, tradition, culture clash, immigration, women\'s football (soccer), sports, football (soccer), family, lgbt, woman director, british asian_Sometimes, to follow your dreams... you\'ve got to bend the rules!'

In [263]:
movies_cleaned['keywords']

0         [{'id': 6075, 'name': 'sports'}, {'id': 10191,...
1                                                        []
2                    [{'id': 215272, 'name': 'megacities'}]
3         [{'id': 970, 'name': 'parent child relationshi...
4         [{'id': 30, 'name': 'individual'}, {'id': 246,...
                                ...                        
252745                                                   []
252746                                                   []
252747                                                   []
252748                                                   []
252749                                                   []
Name: keywords, Length: 252750, dtype: object

In [256]:
movies_df['overview']

1         "Elmo is making a very, very super special sur...
2         "Timo Novotny labels his new project an experi...
3         "Nemo, an adventurous young clownfish, is unex...
4         "Selma, a Czech immigrant on the verge of blin...
5         "In an attempt to pull her family together, Ad...
                                ...                        
252742    "\u201cA Significant Name\u201d tells the stor...
252744    "\u2018Cinegraffic Score\u2019 is a 6 meter co...
252747    "A kaleidoscope of colors and foliage expresse...
252748    "Dr. Heiser explores the connection between UF...
252749    "They do accounting, handle human-resources ma...
Name: overview, Length: 217454, dtype: object

In [257]:
movies_df['tagline']

1                                                       NaN
2                                       A Megacities remix.
3         There are 3.7 trillion fish in the ocean. They...
4                               You don't need eyes to see.
5                    One of the living for one of the dead.
                                ...                        
252742                                                  NaN
252744                                                  NaN
252747                                                  NaN
252748    What is the connection between UFOs, alien abd...
252749                                                  NaN
Name: tagline, Length: 217454, dtype: object

In [258]:
movies_df['omdb_Plot']

1         Elmo is making a super special surprise card f...
2         A remix of images and sounds, using a films or...
3         After his son is captured in the Great Barrier...
4         An Eastern European US immigrant with a love f...
5         In mourning over the tragic drowning of their ...
                                ...                        
252742                                                  NaN
252744                                                  NaN
252747    A year after a devastating flood has killed fi...
252748    Decades after surviving the Nostromo incident,...
252749                                                  NaN
Name: omdb_Plot, Length: 217454, dtype: object